# Valley Fever | Independent environmental-feature investigation

**Purpose:** test whether PM10, rainfall and AZMet measurements improve next-month county forecasts beyond stronger case-history baselines; identify failure modes. This is a follow-up to completed Tier 1 / Tier 2, not a replacement for the old results.

- Exact source: `VF_Environment_Ablation_Input.csv` (357 rows from user-supplied Harsha ZIP); expected SHA256 `aa2233c627cf156af06d48548fec3fbdc0bfc906f6c741b2cafaeaa26236c973`. You may upload the ZIP's `VF_ThreeCounty_Stage2_Model_Input.csv` directly, as long as it has the same contents.
- All 34 configurations have the same training and evaluation month windows. Six-month lag calculation requires removing the first six source rows in **every** arm (113 per county). Nothing is changed in the ADHS case counts.
- Model-selection period **2019–2021** only. Subsequent **2022–2023**, **2024**, and **provisional 2025** observations do not choose the rainfall lag. Importantly, the project had already used 2024 in earlier experiments, so this is **not** a pristine independent test across the entire capstone.
- Model comparisons: Ridge, original model families (NB2 for Maricopa, pooled XGBoost for Pinal/Pima), previous-month/seasonal/three-month simple baselines, simulated delayed case counts, and excluding Q1 2024 targets from 2025 training.
- **Use caution:** case histories are event-date figures, which may not be available promptly; weather data release times not independently verified. 2025 is provisional. PM10 measures particulate dust, not fungal spores. PRISM data are centroid values.

**Instructions:** Runtime → Run all. Upload the combined stage2 CSV if prompted. Outputs are saved to a ZIP at the end. The separately supplied Excel workbook is a formatted report of a verified local run, not something to combine with different-version Colab numbers.


In [ ]:
from pathlib import Path
import os, shutil, hashlib
source=Path('/mnt/data/VF_Environment_Ablation_Input.csv')
source.parent.mkdir(exist_ok=True,parents=True)
if not source.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Put VF_Environment_Ablation_Input.csv in /mnt/data, or run in Google Colab to upload the source CSV')
    uploaded=files.upload()
    if len(uploaded)!=1:raise RuntimeError('Upload exactly one CSV from your combined stage 2 ZIP')
    filename=next(iter(uploaded))
    shutil.copyfile(filename,source)
sha=hashlib.sha256(source.read_bytes()).hexdigest()
if sha!='aa2233c627cf156af06d48548fec3fbdc0bfc906f6c741b2cafaeaa26236c973':
    raise ValueError('Input CSV differs from the audited 357-row original. Do not run tests on an unverified different source. Found SHA256: '+sha)
os.environ['VF_INPUT']=str(source)
os.environ['VF_OUTPUT']='/mnt/data/VF_Followup_Independent_Tests'
print('Verified input:',source, 'SHA256:',sha)


## 1. Full historical selection and future-year comparisons
Run the full predeclared feature sweep; candidate selection only uses 2019–2021.

In [ ]:
from pathlib import Path
import runpy
script_path=Path('/mnt/data/_vf_deepcheck_run.py')
script_path.write_text('"""Auditable Valley Fever lag/timing experiment. Input 357-row stage2 CSV; no case edits.\nCSV machine learning analysis only; spreadsheets are authored separately using artifact_tool.\n"""\nimport os, sys, hashlib, json, warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport sklearn, xgboost, statsmodels\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom xgboost import XGBRegressor\nfrom statsmodels.discrete.discrete_model import NegativeBinomial\nfrom statsmodels.tools.tools import add_constant\n\nSRC=Path(os.environ.get(\'VF_INPUT\',\'/mnt/data/VF_Environment_Ablation_Input.csv\'))\nOUT=Path(os.environ.get(\'VF_OUTPUT\',\'/mnt/data/VF_Followup_Independent_Tests\'));OUT.mkdir(parents=True,exist_ok=True)\nSEED=42\nRNG=np.random.default_rng(SEED)\nCASE_FULL=[\'case_log0\',\'case_log1\',\'case_log2\',\'season_log\',\'month_sin\',\'month_cos\']\nCASE_DELAYED=[\'case_log1\',\'case_log2\',\'season_log\',\'month_sin\',\'month_cos\']\nFIELDS={\'PM10\':\'EPA_PM10_mean_ugm3\',\'RAIN\':\'PRISM_rain_total_in\',\'SOIL\':\'AZMET_soil20_C\',\'HUMIDITY\':\'AZMET_RH_pct\',\'WIND\':\'AZMET_wind_mean_ms\',\'VPD\':\'AZMET_VPD_kPa\'}\n\n\ndef load():\n    src=pd.read_csv(SRC)\n    assert len(src)==357 and set(src.County)=={\'Maricopa\',\'Pinal\',\'Pima\'}\n    src[\'date\']=pd.to_datetime(src.Target_Month+\'-01\'); src[\'year\']=src.date.dt.year\n    assert src.duplicated([\'County\',\'Target_Month\']).sum()==0\n    assert (pd.to_datetime(src.Feature_Month+\'-01\')+pd.offsets.MonthBegin(1)).eq(src.date).all()\n    assert (src.groupby(\'County\').size()==119).all()\n    for col in [\'Target_Cases_Next_Month\',\'VF_cases_current\',\'VF_cases_lag1\',\'VF_cases_lag2\',\'Seasonal_Baseline_Cases\']:\n        src[col]=pd.to_numeric(src[col]); assert src[col].notna().all() and (src[col]>=0).all()\n    src=src.sort_values([\'County\',\'date\']).reset_index(drop=True)\n    for i,col in enumerate([\'VF_cases_current\',\'VF_cases_lag1\',\'VF_cases_lag2\']):\n        src[f\'case_log{i}\']=np.log1p(src[col])\n    src[\'season_log\']=np.log1p(src.Seasonal_Baseline_Cases)\n    src[\'month_sin\']=np.sin(2*np.pi*src.date.dt.month/12)\n    src[\'month_cos\']=np.cos(2*np.pi*src.date.dt.month/12)\n    # Check that case feature lags follow observed previous months on input rows.\n    for county,g in src.groupby(\'County\'):\n        assert (g.VF_cases_lag1.iloc[1:].to_numpy()==g.VF_cases_current.iloc[:-1].to_numpy()).all(),f\'{county} lag1 mismatch\'\n        assert (g.VF_cases_lag2.iloc[2:].to_numpy()==g.VF_cases_current.iloc[:-2].to_numpy()).all(),f\'{county} lag2 mismatch\'\n        # Previous month case is the feature-month actual, except at earliest offset.\n        assert (g.VF_cases_current.iloc[1:].to_numpy()==g.Target_Cases_Next_Month.iloc[:-1].to_numpy()).all(),f\'{county} target mismatch\'\n    for name,base in FIELDS.items():\n        for lag in [0,1,2,3]:\n            orig=f\'{base}_current\' if lag==0 else f\'{base}_lag{lag}\'\n            if orig not in src.columns:raise ValueError(orig)\n            vals=pd.to_numeric(src[orig],errors=\'raise\').astype(float)\n            if name in (\'PM10\',\'RAIN\'):\n                if (vals<0).any():raise ValueError(f\'negative {orig}\')\n                vals=np.log1p(vals)\n            src[f\'{name}_{lag}\']=vals\n        vals=src.groupby(\'County\')[f\'{name}_0\'].shift(6)\n        src[f\'{name}_6\']=vals\n        # Compare lag1/2/3 existing values to lagged observed values, except initial missing months\n        for lag in [1,2,3]:\n            shifted=src.groupby(\'County\')[f\'{name}_0\'].shift(lag)\n            good=shifted.notna()\n            err=(shifted[good]-src.loc[good,f\'{name}_{lag}\']).abs().max()\n            if err>0.001: print(f\'WARNING: {name} stored lag{lag} differs from shifted source, max={err}\')\n    # ALL configs use same rows, even models without lag6, to keep training and test windows identical\n    src=src[src.groupby(\'County\').cumcount()>=6].copy().reset_index(drop=True)\n    assert src.groupby(\'County\').size().eq(113).all()\n    assert src[[f\'{name}_{lag}\' for name in FIELDS for lag in [0,1,2,3,6]]].notna().all().all()\n    return src\n\nCONFIGS={\'CASE_ONLY\':[]}\nfor lag in [0,1,2,3,6]:\n    for group,names in [(\'PM10\',[\'PM10\']),(\'RAIN\',[\'RAIN\']),(\'SOIL\',[\'SOIL\']),(\'PM10_RAIN\',[\'PM10\',\'RAIN\']),(\'DUST_RAIN_SOIL\',[\'PM10\',\'RAIN\',\'SOIL\']),(\'ALL6\',list(FIELDS))]:\n        CONFIGS[f\'{group}_L{lag}\']=[f\'{n}_{lag}\' for n in names]\nfor group,names in [(\'PM10\',[\'PM10\']),(\'RAIN\',[\'RAIN\']),(\'DUST_RAIN_SOIL\',[\'PM10\',\'RAIN\',\'SOIL\'])]:\n    CONFIGS[f\'{group}_L0to3\']=[f\'{n}_{i}\' for n in names for i in [0,1,2,3]]\nassert len(CONFIGS)==34\n# Selection intentionally limits complexity to <=3 environmental features. All 33 variants still logged.\nSELECTABLE={k:v for k,v in CONFIGS.items() if k!=\'CASE_ONLY\' and len(v)<=3}\n\n\ndef predict_ridge(train,test,cols):\n    m=make_pipeline(StandardScaler(),Ridge(alpha=10.0))\n    m.fit(train[cols].to_numpy(dtype=float),train.Target_Cases_Next_Month.to_numpy(dtype=float))\n    return np.maximum(0,m.predict(test[cols].to_numpy(dtype=float)))\n\ndef metrics_from_predictions(preds,groupcols):\n    z=preds.copy()\n    z[\'err\']=np.abs(z.actual-z.pred)\n    z[\'sqr\']=(z.actual-z.pred)**2\n    res=z.groupby(groupcols,dropna=False,sort=False).agg(n=(\'err\',\'size\'),mae=(\'err\',\'mean\'),rmse=(\'sqr\',lambda x:np.sqrt(x.mean())),bias=(\'pred\',lambda x: 0)).reset_index()\n    # bias added from direct residuals\n    z[\'signed_error\']=z.pred-z.actual\n    means=z.groupby(groupcols,dropna=False,sort=False).signed_error.mean().reset_index(name=\'bias_calc\')\n    res=res.drop(columns=\'bias\').merge(means,on=groupcols).rename(columns={\'bias_calc\':\'bias\'})\n    return res\n\ndef run_ridge(d):\n    rows=[]\n    for county,cdf in d.groupby(\'County\',sort=False):\n        for year in range(2019,2026):\n            tr=cdf[cdf.year<year]; te=cdf[cdf.year==year]\n            if year==2025:assert all(tr.year<=2024)\n            for cfg,env in CONFIGS.items():\n                pr=predict_ridge(tr,te,CASE_FULL+env)\n                for r,p in zip(te.itertuples(index=False),pr):\n                    rows.append(dict(county=county,year=year,target_month=r.Target_Month,feature_month=r.Feature_Month,config=cfg,env_n_features=len(env),actual=float(r.Target_Cases_Next_Month),pred=float(p),early_q1_2024=r.Early_2024_Surveillance_Flag,train_n=len(tr),period=(\'SELECT_2019_2021\' if year<=2021 else \'CONFIRM_2022_2023\' if year<=2023 else \'RETRO_2024\' if year==2024 else \'PROVISIONAL_2025\')))\n    p=pd.DataFrame(rows);p[\'abs_error\']=abs(p.actual-p.pred);p[\'signed_error\']=p.pred-p.actual\n    m=metrics_from_predictions(p,[\'county\',\'config\',\'period\'])\n    ym=metrics_from_predictions(p,[\'county\',\'config\',\'year\'])\n    # Candidate selection uses ONLY 2019-2021; no 2022+, no 2024.\n    sel=m[m.period==\'SELECT_2019_2021\'].copy()\n    sels=[]\n    for county,g in sel.groupby(\'county\'):\n        case_mae=float(g.loc[g.config==\'CASE_ONLY\',\'mae\'].iloc[0])\n        eligible=g[g.config.isin(SELECTABLE)].sort_values([\'mae\',\'config\'])\n        best=eligible.iloc[0]\n        sels.append(dict(county=county,selected_config=best.config,env_feature_count=len(CONFIGS[best.config]),selection_months=36,selection_caseonly_mae=case_mae,selection_selected_mae=float(best.mae),selection_delta=case_mae-float(best.mae),candidate_count=len(SELECTABLE),selection_note=\'Selected from 2019-2021 MAE only; 2022 onward not used for selection; <=3 environment predictors\'))\n    s=pd.DataFrame(sels)\n    selected=p.merge(s[[\'county\',\'selected_config\']],on=\'county\')\n    selected=selected[(selected.config==\'CASE_ONLY\')|(selected.config==selected.selected_config)].copy()\n    selected[\'arm\']=np.where(selected.config==\'CASE_ONLY\',\'CASE_ONLY\',\'SELECTED_ENV\')\n    evals=metrics_from_predictions(selected,[\'county\',\'period\',\'arm\'])\n    pair=evals.pivot(index=[\'county\',\'period\'],columns=\'arm\',values=\'mae\').reset_index()\n    pair[\'delta_mae_case_minus_env\']=pair[\'CASE_ONLY\']-pair[\'SELECTED_ENV\']\n    pair[\'pct_gain_vs_case\']=100*pair.delta_mae_case_minus_env/pair.CASE_ONLY\n    for section,filter_ in [(\'2024_APR_DEC\', (selected.year==2024)&(pd.to_datetime(selected.target_month+\'-01\').dt.month>=4)),(\'2024_JAN_MAR\',(selected.year==2024)&(pd.to_datetime(selected.target_month+\'-01\').dt.month<=3))]:\n        vals=metrics_from_predictions(selected[filter_],[\'county\',\'arm\'])\n        pp=vals.pivot(index=\'county\',columns=\'arm\',values=\'mae\').reset_index()\n        pp[\'period\']=section;pp[\'delta_mae_case_minus_env\']=pp.CASE_ONLY-pp.SELECTED_ENV;pp[\'pct_gain_vs_case\']=100*pp.delta_mae_case_minus_env/pp.CASE_ONLY\n        pair=pd.concat([pair,pp[pair.columns]],ignore_index=True)\n    # Simpler baselines evaluated on exactly the same months as Ridge test/selection, per county.\n    baselines=[]\n    for r in d[d.year>=2019].itertuples(index=False):\n        for name,pred in [(\'SAME_MONTH_LAST_YEAR\',r.Seasonal_Baseline_Cases),(\'PREVIOUS_MONTH\',r.VF_cases_current),(\'THREE_MONTH_AVERAGE\',(r.VF_cases_current+r.VF_cases_lag1+r.VF_cases_lag2)/3)]:\n            baselines.append(dict(county=r.County,year=r.year,target_month=r.Target_Month,baseline=name,actual=float(r.Target_Cases_Next_Month),pred=float(pred),period=(\'SELECT_2019_2021\' if r.year<=2021 else \'CONFIRM_2022_2023\' if r.year<=2023 else \'RETRO_2024\' if r.year==2024 else \'PROVISIONAL_2025\')))\n    b=pd.DataFrame(baselines)\n    bm=metrics_from_predictions(b,[\'county\',\'baseline\',\'period\'])\n    return p,m,ym,s,selected,pair,b,bm,sel\n\n\ndef confirm_prior_families(d,selection):\n    """A separate robustness check: uses previously established NB2 for Maricopa, pooled XGB for Pinal/Pima.\n    NB2 + XGB fit to the same CASE_FULL predictor fields as Ridge screening; not same as original old exact model.\n    Fixed selected config per county selected only on 2019-21 Ridge screening. No tuning on 2022/2024.\n    """\n    out=[]\n    xgbcfg=dict(n_estimators=190,max_depth=3,learning_rate=0.04,min_child_weight=5,subsample=0.8,colsample_bytree=0.8,reg_lambda=15,random_state=42,n_jobs=1,verbosity=0,objective=\'reg:squarederror\',tree_method=\'hist\')\n    for r in selection.itertuples(index=False):\n        county=r.county\n        for year in range(2019,2026):\n            for arm,config in [(\'CASE_ONLY\',\'CASE_ONLY\'),(\'SELECTED_ENV\',r.selected_config)]:\n                fields=CASE_FULL+CONFIGS[config]\n                if county==\'Maricopa\':\n                    tr=d[(d.County==county)&(d.year<year)];te=d[(d.County==county)&(d.year==year)]\n                    train_x=tr[fields].to_numpy(dtype=float); test_x=te[fields].to_numpy(dtype=float)\n                    # Standardized covariates for numerical stability, statsmodels NB2 includes estimated alpha.\n                    scale=StandardScaler().fit(train_x)\n                    X=add_constant(scale.transform(train_x),has_constant=\'add\')\n                    Xt=add_constant(scale.transform(test_x),has_constant=\'add\')\n                    try:\n                        with warnings.catch_warnings():\n                            warnings.simplefilter(\'ignore\')\n                            m=NegativeBinomial(tr.Target_Cases_Next_Month.to_numpy(dtype=float), X, loglike_method=\'nb2\').fit(method=\'bfgs\',maxiter=170,disp=False)\n                        if not m.mle_retvals.get(\'converged\',False): raise RuntimeError(\'NB2 non-convergence\')\n                        pred=np.asarray(m.predict(Xt))\n                        family=\'Maricopa NB2\'\n                    except Exception as ex:\n                        print(\'CONFIRM NB2 FAILED\',county,year,arm,repr(ex))\n                        pred=np.full(len(te),np.nan)\n                        family=\'Maricopa NB2 FAILED\'\n                else:\n                    tr=d[d.year<year];te=d[(d.County==county)&(d.year==year)]\n                    xtr=tr[fields].copy();xte=te[fields].copy()\n                    for c in [\'Maricopa\',\'Pinal\']:\n                        xtr[\'county_\'+c]=(tr.County==c).astype(int).to_numpy()\n                        xte[\'county_\'+c]=(te.County==c).astype(int).to_numpy()\n                    model=XGBRegressor(**xgbcfg)\n                    model.fit(xtr.to_numpy(dtype=float),tr.Target_Cases_Next_Month.to_numpy(dtype=float))\n                    pred=np.maximum(0,model.predict(xte.to_numpy(dtype=float)))\n                    family=\'Pooled XGB_medium\'\n                for rr,p in zip(te.itertuples(index=False),pred):\n                    out.append(dict(county=county,year=year,arm=arm,config=config,model_family=family,target_month=rr.Target_Month,actual=float(rr.Target_Cases_Next_Month),pred=float(p),period=(\'SELECT_2019_2021\' if year<=2021 else \'CONFIRM_2022_2023\' if year<=2023 else \'RETRO_2024\' if year==2024 else \'PROVISIONAL_2025\')))\n    z=pd.DataFrame(out)\n    z=z.dropna(subset=[\'pred\'])\n    sums=metrics_from_predictions(z,[\'county\',\'period\',\'arm\',\'model_family\'])\n    comp=sums.pivot(index=[\'county\',\'period\',\'model_family\'],columns=\'arm\',values=\'mae\').reset_index()\n    comp[\'delta_mae_case_minus_env\']=comp.CASE_ONLY-comp.SELECTED_ENV\n    comp[\'pct_gain_vs_case\']=100*comp.delta_mae_case_minus_env/comp.CASE_ONLY\n    return z,sums,comp\n\n\ndef delayed_case_test(d,s):\n    out=[]\n    for r in s.itertuples(index=False):\n        county=r.county; cfg=r.selected_config\n        sub=d[d.County==county]\n        for year in range(2022,2026):\n            tr=sub[sub.year<year];te=sub[sub.year==year]\n            for arm,fields in [(\'DELAYED_CASE_ONLY\',CASE_DELAYED),(\'DELAYED_CASE_PLUS_SELECTED_ENV\',CASE_DELAYED+CONFIGS[cfg])]:\n                pred=predict_ridge(tr,te,fields)\n                for rr,p in zip(te.itertuples(index=False),pred):\n                    out.append(dict(county=county,year=year,arm=arm,selected_config=cfg,target_month=rr.Target_Month,actual=float(rr.Target_Cases_Next_Month),pred=float(p)))\n    z=pd.DataFrame(out)\n    z[\'period\']=np.select([z.year<=2023,z.year==2024,z.year==2025],[\'CONFIRM_2022_2023\',\'RETRO_2024\',\'PROVISIONAL_2025\'],default=\'OTHER\')\n    m=metrics_from_predictions(z,[\'county\',\'period\',\'arm\'])\n    comp=m.pivot(index=[\'county\',\'period\'],columns=\'arm\',values=\'mae\').reset_index()\n    comp[\'delta_mae_case_minus_env\']=comp.DELAYED_CASE_ONLY-comp.DELAYED_CASE_PLUS_SELECTED_ENV\n    return z,m,comp\n\n\ndef write(d):\n    p,m,ym,s,selected,pair,b,bm,sel=run_ridge(d)\n    f,fs,fc=confirm_prior_families(d,s)\n    delay,dm,dc=delayed_case_test(d,s)\n    versions={k:getattr(sys.modules[k],\'__version__\',\'unknown\') for k in [\'numpy\',\'pandas\',\'sklearn\',\'xgboost\',\'statsmodels\']};versions[\'python\']=sys.version.split()[0]\n    versions[\'source_csv_sha256\']=hashlib.sha256(SRC.read_bytes()).hexdigest()\n    versions[\'model_seed\']=SEED;versions[\'model_note\']=\'Ridge alpha=10.0; NB2 estimated dispersion; XGB fixed prior params\'\n    versions[\'source_rows\']=357;versions[\'rows_per_county_after_lag6_restriction\']=113\n    versions[\'lag6_note\']=\'Computed from current environmental observations shifting six feature months per county; first 6 county months removed from ALL arms.\'\n    versions[\'selection_note\']=\'34 configs including case-only. Select lowest MAE over 2019-2021 among <=3 environmental features. No 2022/2023/2024/2025 used for candidate selection.\'\n    files={\'Candidate_Selection_2019_2021.csv\':sel,\'Selected_Configurations.csv\':s,\n           \'Ridge_All_Experiments_Metrics.csv\':m,\'Ridge_All_Experiments_Yearly.csv\':ym,\n           \'Ridge_Selected_Monthly_Predictions.csv\':selected,\'Ridge_Selected_Comparisons.csv\':pair,\n           \'Naive_Baseline_Comparisons.csv\':bm,\'Naive_Baseline_Monthly.csv\':b,\n           \'Prior_Model_Families_Monthly.csv\':f,\'Prior_Model_Families_Comparisons.csv\':fc,\n           \'Delayed_Case_History_Monthly.csv\':delay,\'Delayed_Case_History_Comparisons.csv\':dc}\n    for name,frame in files.items():frame.to_csv(OUT/name,index=False,float_format=\'%.6f\')\n    (OUT/\'Versions_and_Methods.json\').write_text(json.dumps(versions,indent=2))\n    # Qa: add case-history vs environment wins months for each period by county\n    y=selected.pivot(index=[\'county\',\'year\',\'target_month\'],columns=\'arm\',values=[\'actual\',\'pred\',\'abs_error\'])\n    y.columns=[\'_\'.join(t) for t in y.columns];y=y.reset_index()\n    y[\'env_helped_this_month\']=y.abs_error_SELECTED_ENV<y.abs_error_CASE_ONLY\n    y.to_csv(OUT/\'Ridge_Monthly_Paired_Errors.csv\',index=False,float_format=\'%.6f\')\n    print(\'==== DATA INTEGRITY ====\', versions)\n    print(\'==== SELECTED CONFIGURATIONS ====\',s.to_string(index=False))\n    print(\'==== RIDGE SELECTED COMPARISONS ====\',pair.sort_values([\'county\',\'period\']).round(2).to_string(index=False))\n    print(\'==== NAIVE BENCHMARKS ====\',bm[bm.period.isin([\'RETRO_2024\',\'PROVISIONAL_2025\'])].round(2).to_string(index=False))\n    print(\'==== ORIGINAL MODEL FAMILY CHECK ====\',fc.round(2).to_string(index=False))\n    print(\'==== DELAYED CASE CHECK ====\',dc.round(2).to_string(index=False))\n    return locals()\n\nif __name__==\'__main__\':\n    d=load()\n    RESULTS=write(d)\n')
runpy.run_path(str(script_path),run_name='__main__')

## 2. Did unusual 2024 Q1 training targets cause the 2025 failures?
Compare training on all of 2024, excluding Q1 2024 targets, or stopping at 2023. All scenarios use the same 2025 observations. Excluding rows does not remove their counts from future lagged predictors.

In [ ]:
from pathlib import Path
import runpy
sensitivity_script=Path('/mnt/data/_vf_extra_sensitivity.py')
sensitivity_script.write_text("import sys, warnings, os\nfrom pathlib import Path\nimport numpy as np,pandas as pd\nsys.path.insert(0,'/mnt/data')\nfrom _vf_deepcheck_run import load, CASE_FULL, CONFIGS, predict_ridge, metrics_from_predictions\nfrom sklearn.preprocessing import StandardScaler\nfrom statsmodels.discrete.discrete_model import NegativeBinomial\nfrom statsmodels.tools.tools import add_constant\nfrom xgboost import XGBRegressor\nOUT=Path(os.environ.get('VF_OUTPUT','/mnt/data/VF_Followup_Independent_Tests'))\nd=load(); s=pd.read_csv(OUT/'Selected_Configurations.csv')\nres=[]\nfor row in s.itertuples(index=False):\n  county=row.county;cfg=row.selected_config\n  test=d[(d.year==2025)&(d.County==county)]\n  for scen in ['FULL_2024','EXCLUDE_Q1_2024_TARGETS','ONLY_2023_OR_EARLIER']:\n    train=d[d.year<2025].copy()\n    if scen=='EXCLUDE_Q1_2024_TARGETS':train=train[train.Early_2024_Surveillance_Flag!='YES']\n    if scen=='ONLY_2023_OR_EARLIER':train=train[train.year<=2023]\n    for arm,env in [('CASE_ONLY',[]),('SELECTED_ENV',CONFIGS[cfg])]:\n      cols=CASE_FULL+env\n      tr=train[train.County==county]\n      preds=predict_ridge(tr,test,cols)\n      for r,p in zip(test.itertuples(index=False),preds):res.append(dict(county=county,scenario=scen,model_family='Ridge',arm=arm,target_month=r.Target_Month,actual=float(r.Target_Cases_Next_Month),pred=float(p),train_n=len(tr)))\n      if county=='Maricopa':\n        scale=StandardScaler().fit(tr[cols].to_numpy(dtype=float));X=add_constant(scale.transform(tr[cols]),has_constant='add');Xt=add_constant(scale.transform(test[cols]),has_constant='add')\n        try:\n          with warnings.catch_warnings():\n            warnings.simplefilter('ignore')\n            m=NegativeBinomial(tr.Target_Cases_Next_Month.to_numpy(dtype=float),X,loglike_method='nb2').fit(method='bfgs',maxiter=170,disp=False)\n          if not m.mle_retvals.get('converged',False):raise RuntimeError('nonconverged')\n          preds=np.asarray(m.predict(Xt))\n          for r,p in zip(test.itertuples(index=False),preds):res.append(dict(county=county,scenario=scen,model_family='NB2',arm=arm,target_month=r.Target_Month,actual=float(r.Target_Cases_Next_Month),pred=float(p),train_n=len(tr)))\n        except Exception as ex:print('NB2 FAILED',scen,arm,repr(ex))\n      else:\n        xtr=train[cols].copy();xte=test[cols].copy()\n        for c in ['Maricopa','Pinal']:\n          xtr['county_'+c]=(train.County==c).astype(int).to_numpy();xte['county_'+c]=(test.County==c).astype(int).to_numpy()\n        m=XGBRegressor(n_estimators=190,max_depth=3,learning_rate=.04,min_child_weight=5,subsample=.8,colsample_bytree=.8,reg_lambda=15,random_state=42,n_jobs=1,verbosity=0,objective='reg:squarederror',tree_method='hist')\n        m.fit(xtr.to_numpy(dtype=float),train.Target_Cases_Next_Month.to_numpy(dtype=float))\n        for r,p in zip(test.itertuples(index=False),np.maximum(0,m.predict(xte.to_numpy(dtype=float)))):res.append(dict(county=county,scenario=scen,model_family='Pooled_XGB_medium',arm=arm,target_month=r.Target_Month,actual=float(r.Target_Cases_Next_Month),pred=float(p),train_n=len(train)))\nz=pd.DataFrame(res);m=metrics_from_predictions(z,['county','scenario','model_family','arm']);p=m.pivot(index=['county','scenario','model_family'],columns='arm',values='mae').reset_index();p['delta_mae_case_minus_env']=p.CASE_ONLY-p.SELECTED_ENV\nz.to_csv(OUT/'2025_Training_Sensitivity_Monthly.csv',index=False,float_format='%.6f')\np.to_csv(OUT/'2025_Training_Sensitivity_Comparisons.csv',index=False,float_format='%.6f')\nprint(p.round(2).to_string(index=False))\n")
runpy.run_path(str(sensitivity_script),run_name='__main__')

## 3. Download actual rerun outputs
Saved metrics are not hard-coded; the ZIP contains all generated CSVs and the software-version manifest. The main Excel report was created separately from the verified reference run.

In [ ]:
from pathlib import Path
import zipfile
result_dir=Path('/mnt/data/VF_Followup_Independent_Tests')
zip_path=Path('/mnt/data/VF_Independent_Test_Colab_Outputs.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(result_dir.glob('*')):
        if f.is_file():z.write(f,arcname=f.name)
print('Result files:',len(list(result_dir.glob('*'))),'\nZIP:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('File saved locally at',zip_path)